In [ ]:
import cv2
import numpy as np
from pathlib import Path
import shutil
from tqdm import tqdm

# Simplified obstacle classes - 6 categories
OBSTACLE_CLASSES = {
    # Vegetation (trees, bushes, plants)
    'tree': {'seg_id': 19, 'yolo_id': 0},
    'bald-tree': {'seg_id': 20, 'yolo_id': 0},
    'vegetation': {'seg_id': 8, 'yolo_id': 0},
    
    # Structures (walls, buildings, roofs)
    'wall': {'seg_id': 10, 'yolo_id': 1},
    'roof': {'seg_id': 9, 'yolo_id': 1},
    'window': {'seg_id': 11, 'yolo_id': 1},
    'door': {'seg_id': 12, 'yolo_id': 1},
    
    # Fences
    'fence': {'seg_id': 13, 'yolo_id': 2},
    'fence-pole': {'seg_id': 14, 'yolo_id': 2},
    
    # Living things
    'person': {'seg_id': 15, 'yolo_id': 3},
    'dog': {'seg_id': 16, 'yolo_id': 3},
    
    # Vehicles
    'car': {'seg_id': 17, 'yolo_id': 4},
    'bicycle': {'seg_id': 18, 'yolo_id': 4},
    
    # Generic obstacle
    'obstacle': {'seg_id': 22, 'yolo_id': 5},
    'rocks': {'seg_id': 6, 'yolo_id': 5},
}

YOLO_CLASS_NAMES = [
    'vegetation',  # 0 (trees, bushes, plants)
    'structure',   # 1 (walls, buildings, roofs)
    'fence',       # 2 (fences and poles)
    'living',      # 3 (people and animals)
    'vehicle',     # 4 (cars, bikes)
    'obstacle',    # 5 (generic obstacles, rocks)
]

# Color mappings for semantic segmentation
# Based on standard semantic drone dataset colors
SEMANTIC_COLORS = {
    0: (0, 0, 0),         # unlabeled
    1: (128, 64, 128),    # paved-area
    2: (130, 76, 0),      # dirt
    3: (0, 102, 0),       # grass
    4: (112, 103, 87),    # gravel
    5: (28, 42, 168),     # water
    6: (48, 41, 30),      # rocks
    7: (0, 50, 89),       # pool
    8: (107, 142, 35),    # vegetation
    9: (70, 70, 70),      # roof
    10: (102, 102, 156),  # wall
    11: (254, 228, 12),   # window
    12: (254, 148, 12),   # door
    13: (190, 153, 153),  # fence
    14: (153, 153, 153),  # fence-pole
    15: (255, 22, 96),    # person
    16: (102, 51, 0),     # dog
    17: (9, 143, 150),    # car
    18: (119, 11, 32),    # bicycle
    19: (51, 51, 0),      # tree
    20: (190, 250, 190),  # bald-tree
    21: (112, 150, 146),  # ar-marker
    22: (2, 135, 115),    # obstacle
    23: (255, 0, 0),      # conflicting
}

def create_color_to_class_map():
    """Create mapping from RGB color to class ID"""
    color_to_class = {}
    for class_id, color in SEMANTIC_COLORS.items():
        color_to_class[color] = class_id
    return color_to_class

def mask_to_bboxes(mask_path, img_width, img_height, color_to_class):
    """Convert segmentation mask to YOLO bounding boxes"""
    
    mask = cv2.imread(str(mask_path))
    if mask is None:
        return []
    
    mask_rgb = cv2.cvtColor(mask, cv2.COLOR_BGR2RGB)
    h, w = mask.shape[:2]
    
    bboxes = []
    
    # Process each obstacle class
    for class_name, info in OBSTACLE_CLASSES.items():
        seg_id = info['seg_id']
        yolo_id = info['yolo_id']
        
        # Get color for this semantic class
        if seg_id not in SEMANTIC_COLORS:
            continue
        
        color = SEMANTIC_COLORS[seg_id]
        
        # Create binary mask for this class
        class_mask = np.all(mask_rgb == color, axis=-1).astype(np.uint8) * 255
        
        if class_mask.sum() == 0:
            continue
        
        # Find contours
        contours, _ = cv2.findContours(class_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        
        for contour in contours:
            area = cv2.contourArea(contour)
            if area < 400:  # Skip tiny objects
                continue
            
            x, y, w_box, h_box = cv2.boundingRect(contour)
            
            if w_box < 15 or h_box < 15:
                continue
            
            # Convert to YOLO format (normalized)
            x_center = (x + w_box / 2) / img_width
            y_center = (y + h_box / 2) / img_height
            width_norm = w_box / img_width
            height_norm = h_box / img_height
            
            # Clamp
            x_center = np.clip(x_center, 0, 1)
            y_center = np.clip(y_center, 0, 1)
            width_norm = np.clip(width_norm, 0, 1)
            height_norm = np.clip(height_norm, 0, 1)
            
            bboxes.append([yolo_id, x_center, y_center, width_norm, height_norm])
    
    return bboxes

def convert_dataset():
    """Main conversion function"""
    
    print("\n" + "="*70)
    print("SEMANTIC DRONE DATASET → YOLO CONVERTER")
    print("="*70)
    
    dataset_path = Path('dataset/semantic_drone_dataset')
    img_dir = dataset_path / 'original_images'
    mask_dir = dataset_path / 'label_images_semantic'
    
    if not img_dir.exists() or not mask_dir.exists():
        print(f"❌ Could not find directories:")
        print(f"   Images: {img_dir}")
        print(f"   Masks: {mask_dir}")
        return
    
    print(f"✅ Images: {img_dir}")
    print(f"✅ Masks: {mask_dir}")
    
    # Create color mapping
    color_to_class = create_color_to_class_map()
    
    # Create output structure
    output_base = Path('SemanticDrone-YOLO')
    
    # Get all images
    image_files = sorted(list(img_dir.glob('*.jpg')) + list(img_dir.glob('*.png')))
    
    if not image_files:
        print(f"❌ No images found in {img_dir}")
        return
    
    print(f"\n📁 Found {len(image_files)} images")
    
    # Split train/val (80/20)
    import random
    random.seed(42)
    random.shuffle(image_files)
    
    split_idx = int(len(image_files) * 0.8)
    splits = {
        'train': image_files[:split_idx],
        'val': image_files[split_idx:]
    }
    
    total_objects = 0
    
    for split_name, img_list in splits.items():
        print(f"\n{'='*70}")
        print(f"Processing {split_name.upper()} ({len(img_list)} images)")
        print(f"{'='*70}")
        
        out_img_dir = output_base / split_name / 'images'
        out_lbl_dir = output_base / split_name / 'labels'
        
        out_img_dir.mkdir(parents=True, exist_ok=True)
        out_lbl_dir.mkdir(parents=True, exist_ok=True)
        
        split_objects = 0
        processed = 0
        
        for img_path in tqdm(img_list, desc=f"Converting {split_name}"):
            # Find corresponding mask
            mask_path = mask_dir / img_path.name
            if not mask_path.exists():
                mask_path = mask_dir / (img_path.stem + '.png')
            
            if not mask_path.exists():
                continue
            
            # Get image dimensions
            img = cv2.imread(str(img_path))
            if img is None:
                continue
            
            h, w = img.shape[:2]
            
            # Convert mask to bboxes
            bboxes = mask_to_bboxes(mask_path, w, h, color_to_class)
            
            if not bboxes:
                continue
            
            # Save labels
            label_path = out_lbl_dir / (img_path.stem + '.txt')
            with open(label_path, 'w') as f:
                for bbox in bboxes:
                    f.write(' '.join(map(str, bbox)) + '\n')
            
            # Copy image
            shutil.copy(img_path, out_img_dir / img_path.name)
            
            split_objects += len(bboxes)
            processed += 1
        
        print(f"✅ {split_name}: {processed} images, {split_objects} objects")
        total_objects += split_objects
    
    print("\n" + "="*70)
    print("CONVERSION COMPLETE!")
    print("="*70)
    print(f"📦 Total objects detected: {total_objects}")
    
    # Create YAML config
    yaml_content = f"""# Semantic Drone Dataset Configuration
path: SemanticDrone-YOLO
train: train/images
val: val/images

# Classes
nc: {len(YOLO_CLASS_NAMES)}
names: {YOLO_CLASS_NAMES}
"""
    
    with open('SemanticDrone.yaml', 'w') as f:
        f.write(yaml_content)
    
    print("\n✅ Created SemanticDrone.yaml")
    print("\n🚀 Ready to train!")
    print("\nNext step: Run 'python train_semantic.py'")

if __name__ == "__main__":
    try:
        convert_dataset()
    except Exception as e:
        print(f"\n❌ Error: {e}")
        import traceback
        traceback.print_exc()

In [3]:
# ===== SETUP & IMPORTS =====
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import shutil
from tqdm.notebook import tqdm
from IPython.display import Video, HTML, display, Image as IPImage
import warnings
warnings.filterwarnings('ignore')

# Check if running in Jupyter
try:
    get_ipython()
    IN_JUPYTER = True
except:
    IN_JUPYTER = False

print("✅ Imports successful!")
print(f"📓 Running in Jupyter: {IN_JUPYTER}")

✅ Imports successful!
📓 Running in Jupyter: True


In [4]:
# ===== DATASET CONVERSION: SEGMENTATION TO YOLO =====

print("="*70)
print("STEP 1: CONVERTING SEMANTIC DRONE DATASET TO YOLO FORMAT")
print("="*70)

# Obstacle class mapping
OBSTACLE_CLASSES = {
    'tree': {'seg_id': 19, 'yolo_id': 0},
    'bald-tree': {'seg_id': 20, 'yolo_id': 0},
    'vegetation': {'seg_id': 8, 'yolo_id': 0},
    'wall': {'seg_id': 10, 'yolo_id': 1},
    'roof': {'seg_id': 9, 'yolo_id': 1},
    'window': {'seg_id': 11, 'yolo_id': 1},
    'door': {'seg_id': 12, 'yolo_id': 1},
    'fence': {'seg_id': 13, 'yolo_id': 2},
    'fence-pole': {'seg_id': 14, 'yolo_id': 2},
    'person': {'seg_id': 15, 'yolo_id': 3},
    'dog': {'seg_id': 16, 'yolo_id': 3},
    'car': {'seg_id': 17, 'yolo_id': 4},
    'bicycle': {'seg_id': 18, 'yolo_id': 4},
    'obstacle': {'seg_id': 22, 'yolo_id': 5},
    'rocks': {'seg_id': 6, 'yolo_id': 5},
}

YOLO_CLASS_NAMES = [
    'vegetation', 'structure', 'fence', 
    'living', 'vehicle', 'obstacle'
]

# Semantic color mappings
SEMANTIC_COLORS = {
    6: (48, 41, 30),      # rocks
    8: (107, 142, 35),    # vegetation
    9: (70, 70, 70),      # roof
    10: (102, 102, 156),  # wall
    11: (254, 228, 12),   # window
    12: (254, 148, 12),   # door
    13: (190, 153, 153),  # fence
    14: (153, 153, 153),  # fence-pole
    15: (255, 22, 96),    # person
    16: (102, 51, 0),     # dog
    17: (9, 143, 150),    # car
    18: (119, 11, 32),    # bicycle
    19: (51, 51, 0),      # tree
    20: (190, 250, 190),  # bald-tree
    22: (2, 135, 115),    # obstacle
}

print(f"📊 Mapping {len(SEMANTIC_COLORS)} semantic classes → {len(YOLO_CLASS_NAMES)} YOLO classes")
print(f"✅ Class mapping loaded")

STEP 1: CONVERTING SEMANTIC DRONE DATASET TO YOLO FORMAT
📊 Mapping 15 semantic classes → 6 YOLO classes
✅ Class mapping loaded


In [5]:
# ===== CONVERSION FUNCTION =====

def mask_to_bboxes(mask_path, img_width, img_height):
    """Convert segmentation mask to YOLO bounding boxes"""
    
    mask = cv2.imread(str(mask_path))
    if mask is None:
        return []
    
    mask_rgb = cv2.cvtColor(mask, cv2.COLOR_BGR2RGB)
    bboxes = []
    
    for class_name, info in OBSTACLE_CLASSES.items():
        seg_id = info['seg_id']
        yolo_id = info['yolo_id']
        
        if seg_id not in SEMANTIC_COLORS:
            continue
        
        color = SEMANTIC_COLORS[seg_id]
        class_mask = np.all(mask_rgb == color, axis=-1).astype(np.uint8) * 255
        
        if class_mask.sum() == 0:
            continue
        
        contours, _ = cv2.findContours(class_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        
        for contour in contours:
            if cv2.contourArea(contour) < 400:
                continue
            
            x, y, w, h = cv2.boundingRect(contour)
            if w < 15 or h < 15:
                continue
            
            x_center = np.clip((x + w/2) / img_width, 0, 1)
            y_center = np.clip((y + h/2) / img_height, 0, 1)
            width_norm = np.clip(w / img_width, 0, 1)
            height_norm = np.clip(h / img_height, 0, 1)
            
            bboxes.append([yolo_id, x_center, y_center, width_norm, height_norm])
    
    return bboxes

print("✅ Conversion function defined")

✅ Conversion function defined


In [6]:
# ===== RUN CONVERSION =====

dataset_path = Path('dataset/semantic_drone_dataset')
img_dir = dataset_path / 'original_images'
mask_dir = dataset_path / 'label_images_semantic'

if not img_dir.exists():
    print("❌ Dataset not found!")
    print(f"Expected path: {img_dir}")
else:
    print(f"✅ Found dataset at: {dataset_path}")
    
    # Get images
    image_files = sorted(list(img_dir.glob('*.jpg')) + list(img_dir.glob('*.png')))
    print(f"📁 Total images: {len(image_files)}")
    
    # Split train/val
    import random
    random.seed(42)
    random.shuffle(image_files)
    split_idx = int(len(image_files) * 0.8)
    
    splits = {
        'train': image_files[:split_idx],
        'val': image_files[split_idx:]
    }
    
    output_base = Path('SemanticDrone-YOLO')
    total_objects = 0
    
    for split_name, img_list in splits.items():
        print(f"\n📦 Processing {split_name}: {len(img_list)} images")
        
        out_img_dir = output_base / split_name / 'images'
        out_lbl_dir = output_base / split_name / 'labels'
        out_img_dir.mkdir(parents=True, exist_ok=True)
        out_lbl_dir.mkdir(parents=True, exist_ok=True)
        
        split_objects = 0
        processed = 0
        
        for img_path in tqdm(img_list, desc=f"{split_name}"):
            mask_path = mask_dir / img_path.name
            if not mask_path.exists():
                mask_path = mask_dir / (img_path.stem + '.png')
            if not mask_path.exists():
                continue
            
            img = cv2.imread(str(img_path))
            if img is None:
                continue
            
            h, w = img.shape[:2]
            bboxes = mask_to_bboxes(mask_path, w, h)
            
            if not bboxes:
                continue
            
            # Save labels
            with open(out_lbl_dir / (img_path.stem + '.txt'), 'w') as f:
                for bbox in bboxes:
                    f.write(' '.join(map(str, bbox)) + '\n')
            
            shutil.copy(img_path, out_img_dir / img_path.name)
            split_objects += len(bboxes)
            processed += 1
        
        print(f"   ✅ {processed} images, {split_objects} objects")
        total_objects += split_objects
    
    print(f"\n🎉 CONVERSION COMPLETE!")
    print(f"📦 Total objects: {total_objects}")
    
    # Create YAML
    yaml_content = f"""path: SemanticDrone-YOLO
train: train/images
val: val/images
nc: {len(YOLO_CLASS_NAMES)}
names: {YOLO_CLASS_NAMES}
"""
    with open('SemanticDrone.yaml', 'w') as f:
        f.write(yaml_content)
    
    print("✅ Created SemanticDrone.yaml")

✅ Found dataset at: dataset\semantic_drone_dataset
📁 Total images: 400

📦 Processing train: 320 images


train:   0%|          | 0/320 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [10]:
# ===== RUN CONVERSION WITH STATUS PRINT (GRAYSCALE MASKS) =====

dataset_path = Path('dataset/semantic_drone_dataset')
img_dir = dataset_path / 'original_images'
mask_dir = dataset_path / 'label_images_semantic'

if not img_dir.exists():
    print("❌ Dataset not found!")
    print(f"Expected path: {img_dir}")
else:
    print(f"✅ Found dataset at: {dataset_path}")

# Get images
image_files = sorted(list(img_dir.glob('*.jpg')) + list(img_dir.glob('*.png')))
print(f"📁 Total images: {len(image_files)}")

# Split train/val
import random
random.seed(42)
random.shuffle(image_files)
split_idx = int(len(image_files) * 0.8)

splits = {
    'train': image_files[:split_idx],
    'val': image_files[split_idx:]
}

output_base = Path('SemanticDrone-YOLO')
total_objects = 0

# --- Updated mask_to_bboxes for grayscale masks ---
def mask_to_bboxes(mask_path, img_width, img_height):
    """Convert grayscale segmentation mask to YOLO bounding boxes"""
    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    if mask is None:
        return []

    bboxes = []

    for class_name, info in OBSTACLE_CLASSES.items():
        seg_id = info['seg_id']
        yolo_id = info['yolo_id']

        # Create binary mask for this class
        class_mask = (mask == seg_id).astype(np.uint8) * 255
        if class_mask.sum() == 0:
            continue

        contours, _ = cv2.findContours(class_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        for contour in contours:
            if cv2.contourArea(contour) < 400:
                continue
            x, y, w, h = cv2.boundingRect(contour)
            if w < 15 or h < 15:
                continue

            x_center = np.clip((x + w/2) / img_width, 0, 1)
            y_center = np.clip((y + h/2) / img_height, 0, 1)
            width_norm = np.clip(w / img_width, 0, 1)
            height_norm = np.clip(h / img_height, 0, 1)

            bboxes.append([yolo_id, x_center, y_center, width_norm, height_norm])

    return bboxes

# --- Iterate over splits and images ---
for split_name, img_list in splits.items():
    print(f"\n📦 Processing {split_name}: {len(img_list)} images")

    out_img_dir = output_base / split_name / 'images'
    out_lbl_dir = output_base / split_name / 'labels'
    out_img_dir.mkdir(parents=True, exist_ok=True)
    out_lbl_dir.mkdir(parents=True, exist_ok=True)

    split_objects = 0
    processed = 0

    for img_path in tqdm(img_list, desc=f"{split_name}"):
        mask_path = mask_dir / img_path.name
        if not mask_path.exists():
            mask_path = mask_dir / (img_path.stem + '.png')
        if not mask_path.exists():
            print(f"⚠️ Mask not found for {img_path.name}, skipping")
            continue

        img = cv2.imread(str(img_path))
        if img is None:
            print(f"⚠️ Unable to read {img_path.name}, skipping")
            continue

        h, w = img.shape[:2]
        bboxes = mask_to_bboxes(mask_path, w, h)

        if not bboxes:
            print(f"❌ No objects detected in {img_path.name}")
        else:
            print(f"✅ Detected {len(bboxes)} objects in {img_path.name}")

        # Save labels if objects found
        if bboxes:
            with open(out_lbl_dir / (img_path.stem + '.txt'), 'w') as f:
                for bbox in bboxes:
                    f.write(' '.join(map(str, bbox)) + '\n')

        shutil.copy(img_path, out_img_dir / img_path.name)
        split_objects += len(bboxes)
        processed += 1

    print(f"   ✅ {processed} images processed, {split_objects} total objects")
    total_objects += split_objects

print(f"\n🎉 CONVERSION COMPLETE!")
print(f"📦 Total objects across all splits: {total_objects}")

# Create YAML
yaml_content = f"""path: SemanticDrone-YOLO
train: train/images
val: val/images
nc: {len(YOLO_CLASS_NAMES)}
names: {YOLO_CLASS_NAMES}
"""
with open('SemanticDrone.yaml', 'w') as f:
    f.write(yaml_content)

print("✅ Created SemanticDrone.yaml")


✅ Found dataset at: dataset\semantic_drone_dataset
📁 Total images: 400

📦 Processing train: 320 images


train:   0%|          | 0/320 [00:00<?, ?it/s]

✅ Detected 60 objects in 372.jpg
✅ Detected 36 objects in 141.jpg
✅ Detected 188 objects in 228.jpg
✅ Detected 18 objects in 235.jpg
✅ Detected 79 objects in 363.jpg
✅ Detected 78 objects in 331.jpg
✅ Detected 32 objects in 275.jpg
✅ Detected 130 objects in 176.jpg
✅ Detected 19 objects in 383.jpg
✅ Detected 29 objects in 437.jpg
✅ Detected 209 objects in 277.jpg
✅ Detected 54 objects in 455.jpg
✅ Detected 41 objects in 059.jpg
✅ Detected 2 objects in 508.jpg
✅ Detected 13 objects in 521.jpg
✅ Detected 81 objects in 549.jpg
✅ Detected 45 objects in 068.jpg
✅ Detected 112 objects in 316.jpg
✅ Detected 244 objects in 149.jpg
✅ Detected 9 objects in 081.jpg
✅ Detected 98 objects in 410.jpg
✅ Detected 75 objects in 240.jpg
✅ Detected 48 objects in 375.jpg
✅ Detected 94 objects in 119.jpg
✅ Detected 36 objects in 013.jpg
✅ Detected 45 objects in 107.jpg
✅ Detected 19 objects in 290.jpg
✅ Detected 104 objects in 002.jpg
✅ Detected 26 objects in 556.jpg
✅ Detected 32 objects in 563.jpg
✅ Dete

val:   0%|          | 0/80 [00:00<?, ?it/s]

✅ Detected 88 objects in 159.jpg
✅ Detected 127 objects in 260.jpg
✅ Detected 12 objects in 273.jpg
✅ Detected 66 objects in 126.jpg
✅ Detected 38 objects in 266.jpg
✅ Detected 36 objects in 338.jpg
✅ Detected 19 objects in 561.jpg
✅ Detected 19 objects in 531.jpg
✅ Detected 39 objects in 079.jpg
✅ Detected 78 objects in 584.jpg
✅ Detected 84 objects in 513.jpg
✅ Detected 65 objects in 215.jpg
✅ Detected 36 objects in 170.jpg
✅ Detected 227 objects in 040.jpg
✅ Detected 43 objects in 056.jpg
✅ Detected 6 objects in 148.jpg
✅ Detected 83 objects in 440.jpg
✅ Detected 104 objects in 265.jpg
✅ Detected 38 objects in 472.jpg
✅ Detected 28 objects in 478.jpg
✅ Detected 46 objects in 217.jpg
✅ Detected 7 objects in 425.jpg
✅ Detected 39 objects in 062.jpg
✅ Detected 23 objects in 281.jpg
✅ Detected 57 objects in 101.jpg
✅ Detected 158 objects in 413.jpg
✅ Detected 51 objects in 342.jpg
✅ Detected 54 objects in 038.jpg
✅ Detected 57 objects in 194.jpg
✅ Detected 91 objects in 461.jpg
✅ Detect

In [11]:
# ===== TRAIN YOLO MODEL =====

from ultralytics import YOLO

print("="*70)
print("STEP 2: TRAINING YOLO ON SEMANTIC DRONE DATASET")
print("="*70)

model = YOLO('yolov8n.pt')

print("🚀 Starting training...")
print("⏱️  This will take 2-4 hours depending on GPU")
print("📊 Watch the metrics below...\n")

results = model.train(
    data='SemanticDrone.yaml',
    epochs=100,
    imgsz=640,
    batch=8,
    device=0,
    project='semantic_drone_training',
    name='yolov8n_semantic',
    
    # Augmentation
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=15.0,
    translate=0.2,
    scale=0.9,
    shear=5.0,
    flipud=0.5,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.15,
    
    patience=20,
    save=True,
    plots=True,
    verbose=False,  # Less output in notebook
)

print("\n✅ TRAINING COMPLETE!")
print(f"📁 Results saved to: {results.save_dir}")
print(f"🏆 Best model: {results.save_dir}/weights/best.pt")

STEP 2: TRAINING YOLO ON SEMANTIC DRONE DATASET


100%|█████████████████████████████████████████████████████████████████████████████| 6.25M/6.25M [00:02<00:00, 2.60MB/s]


🚀 Starting training...
⏱️  This will take 2-4 hours depending on GPU
📊 Watch the metrics below...

New https://pypi.org/project/ultralytics/8.3.221 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.163  Python-3.10.0 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=SemanticDrone.yaml, degrees=15.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.15, mode=train, model=yolov

100%|█████████████████████████████████████████████████████████████████████████████| 5.35M/5.35M [00:02<00:00, 2.66MB/s]


AMP: checks passed 
train: Fast image access  (ping: 0.20.0 ms, read: 963.6110.1 MB/s, size: 9766.4 KB)


...

    100/100      2.45G      1.682      1.401      1.277        252        640: 100%|██████████| 40/40 [00:28<00:00,  1.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 5/5 [00:01<0

                   all         80       4002      0.565      0.429       0.46      0.262



100 epochs completed in 0.949 hours.
Optimizer stripped from semantic_drone_training\yolov8n_semantic\weights\last.pt, 6.2MB
Optimizer stripped from semantic_drone_training\yolov8n_semantic\weights\best.pt, 6.2MB

Validating semantic_drone_training\yolov8n_semantic\weights\best.pt...
Ultralytics 8.3.163  Python-3.10.0 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
Model summary (fused): 72 layers, 3,006,818 parameters, 0 gradients, 8.1 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 5/5 [00:06<0


                   all         80       4002       0.59      0.426      0.463      0.269
Speed: 1.1ms preprocess, 6.2ms inference, 0.0ms loss, 23.8ms postprocess per image
Results saved to semantic_drone_training\yolov8n_semantic

✅ TRAINING COMPLETE!
📁 Results saved to: semantic_drone_training\yolov8n_semantic
🏆 Best model: semantic_drone_training\yolov8n_semantic/weights/best.pt


In [12]:
# ===== LOAD DRONET MODEL =====

import cv2
import numpy as np
from tensorflow.keras.models import model_from_json
from pathlib import Path
import tensorflow as tf

print("="*70)
print("STEP 3: LOADING DRONET FOR STEERING PREDICTION")
print("="*70)

# Paths
model_json_path = Path(r"of-obstacledetection\DroNeTello\models\DroNet\model_struct.json")
model_weights_path = Path(r"of-obstacledetection\DroNeTello\models\DroNet\model_weights_new_best.h5")

# --- File checks ---
if not model_json_path.exists():
    raise FileNotFoundError(f"❌ Model JSON not found: {model_json_path}")
if not model_weights_path.exists():
    raise FileNotFoundError(f"❌ Model weights not found: {model_weights_path}")

# --- Load model ---
with open(model_json_path, "r") as f:
    dronet = model_from_json(f.read())

dronet.load_weights(model_weights_path)

print("✅ DroNet loaded successfully!")
print(f"   Input shape : {dronet.input_shape}")
print(f"   Output shape: {dronet.output_shape}")

# --- Optional: print model summary ---
dronet.summary()

# --- Preprocessing function ---
def preprocess_dronet(frame):
    """
    Preprocess frame for DroNet:
    - Convert to grayscale
    - Resize to 200x200
    - Normalize to [0,1]
    - Add batch and channel dimensions
    """
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    resized = cv2.resize(gray, (200, 200))
    normalized = resized.astype(np.float32) / 255.0
    return normalized[np.newaxis, :, :, np.newaxis]

print("✅ Preprocessing function defined")

# --- Optional: GPU info ---
print("ℹ️ TensorFlow GPUs available:", len(tf.config.list_physical_devices('GPU')))


STEP 3: LOADING DRONET FOR STEERING PREDICTION
✅ DroNet loaded successfully!
   Input shape : (None, 200, 200, 1)
   Output shape: [(None, 1), (None, 1)]
Model: "model_1"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_1 (InputLayer)        [(None, 200, 200, 1)]        0         []                            
                                                                                                  
 conv2d_1 (Conv2D)           (None, 100, 100, 32)         832       ['input_1[0][0]']             
                                                                                                  
 max_pooling2d_1 (MaxPoolin  (None, 49, 49, 32)           0         ['conv2d_1[0][0]']            
 g2D)                                                                                             
                                     

In [13]:
# ===== LOAD TRAINED YOLO =====

yolo_weights = 'semantic_drone_training/yolov8n_semantic/weights/best.pt'

if Path(yolo_weights).exists():
    yolo = YOLO(yolo_weights)
    print("✅ YOLO model loaded successfully!")
else:
    print("⚠️  YOLO weights not found, using pretrained COCO model")
    yolo = YOLO('yolov8n.pt')

✅ YOLO model loaded successfully!


In [28]:
# ===== SAFETY ASSESSMENT FUNCTION (AGGRESSIVE + STATS) =====

def calculate_safety_from_detections(detections, frame_width, frame_height):
    """
    AGGRESSIVE scoring - if trees are detected, danger should be HIGH
    
    Problem: We were detecting obstacles but danger score stayed near 0
    Solution: Much more aggressive accumulation and scoring
    """
    
    if len(detections.boxes) == 0:
        return 0.0, "SAFE", []
    
    # ===== SIMPLE APPROACH: COUNT OBSTACLES IN PATH =====
    
    # Define FLIGHT PATH (narrow center corridor)
    path_x_min = frame_width * 0.35
    path_x_max = frame_width * 0.65
    path_y_min = frame_height * 0.50
    
    danger_score = 0.0
    dangerous_objects = []
    obstacles_in_path = 0
    
    for box in detections.boxes:
        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
        cls = int(box.cls[0].cpu().numpy())
        conf = float(box.conf[0].cpu().numpy())
        
        cx = (x1 + x2) / 2
        cy = (y1 + y2) / 2
        area = (x2 - x1) * (y2 - y1)
        size_ratio = area / (frame_width * frame_height)
        
        # ===== FILTER 1: Must be in lower half =====
        if cy < path_y_min:
            continue
        
        # ===== FILTER 2: Must be visible (not tiny) =====
        if size_ratio < 0.005:  # 0.5% minimum
            continue
        
        # ===== SMART VEGETATION HANDLING =====
        # Only ignore vegetation if CLEARLY background (edges, small, high up)
        if cls == 0:  # vegetation
            centrality = 1.0 - abs(cx - frame_width/2) / (frame_width/2)
            relative_height = (cy - path_y_min) / (frame_height - path_y_min)
            
            # Ignore ONLY if:
            # - Very small (< 2% of frame) AND
            # - Far from center (< 40% centrality) AND  
            # - In upper portion (< 30% down)
            if size_ratio < 0.02 and centrality < 0.4 and relative_height < 0.3:
                continue
        
        # ===== CHECK IF OBSTACLE BLOCKS FLIGHT PATH =====
        blocks_path = (x1 < path_x_max) and (x2 > path_x_min)  # Overlaps path
        
        if blocks_path:
            obstacles_in_path += 1
        
        # ===== CALCULATE DANGER (AGGRESSIVE) =====
        
        # Proximity factor (0.2 to 1.0)
        proximity = (cy - path_y_min) / (frame_height - path_y_min)
        proximity = max(0.2, min(proximity, 1.0))
        
        # Size factor (0 to 1.0)
        size_factor = min(size_ratio * 15, 1.0)  # Amplified
        
        # Centrality (0.2 to 1.0)
        centrality = 1.0 - abs(cx - frame_width/2) / (frame_width/2)
        centrality = max(0.2, centrality)
        
        # Path blocking bonus
        path_bonus = 1.5 if blocks_path else 1.0
        
        # Class weight
        if cls == 0:    # vegetation
            class_weight = 1.2   # HIGHER weight (was being ignored)
        elif cls == 1:  # structure
            class_weight = 1.5
        elif cls == 3:  # living
            class_weight = 2.0
        elif cls == 4:  # vehicle
            class_weight = 1.5
        else:
            class_weight = 1.0
        
        # AGGRESSIVE FORMULA
        object_danger = (
            proximity * 0.4 +
            size_factor * 0.4 +
            centrality * 0.2
        ) * class_weight * path_bonus * conf
        
        # IMPORTANT: Add FULL object danger (no dampening)
        danger_score += object_danger
        
        dangerous_objects.append({
            'bbox': (x1, y1, x2, y2),
            'class': cls,
            'danger': object_danger,
            'size': size_ratio,
            'blocks_path': blocks_path
        })
    
    # ===== BONUS: Multiple obstacles = exponentially more dangerous =====
    if obstacles_in_path >= 3:
        danger_score *= 1.5  # 50% bonus for cluttered path
    elif obstacles_in_path >= 5:
        danger_score *= 2.0  # 100% bonus for very cluttered
    
    # ===== AMPLIFY OVERALL SCORE =====
    danger_score = min(danger_score * 2.0, 1.0)  # 2.5x MULTIPLIER
    
    # ===== LOWER THRESHOLDS =====
    if danger_score > 0.25:      # LOW threshold
        status = "CRASH RISK!"
    elif danger_score > 0.12:    # VERY LOW threshold
        status = "CAUTION"
    else:
        status = "SAFE"
    
    return danger_score, status, dangerous_objects


print("✅ AGGRESSIVE danger scoring loaded")
print("   Key changes:")
print("   - Vegetation weight: 1.2 (trees ARE obstacles)")
print("   - Path blocking: 1.5x bonus")
print("   - Multiple obstacles: up to 2.0x bonus")
print("   - Overall multiplier: 2.5x")
print("   - CRASH threshold: 0.25 (very sensitive)")
print("   - CAUTION threshold: 0.12")

# ===== QUICK STATS CHECK =====

print("\n" + "="*70)
print("📊 Testing on sample frames to verify sensitivity...")
print("="*70)

# Quick test on a few frames
test_cap = cv2.VideoCapture(video_path)
test_scores = []
test_statuses = []

print("\nSampling 50 frames...")
for i in range(50):
    ret, frame = test_cap.read()
    if not ret:
        break
    
    test_detections = yolo(frame, verbose=False)[0]
    danger, status, objs = calculate_safety_from_detections(
        test_detections, 
        int(test_cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
        int(test_cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    )
    test_scores.append(danger)
    test_statuses.append(status)

test_cap.release()

if test_scores:
    test_scores_arr = np.array(test_scores)
    
    print(f"\n📈 Quick Stats (50 frames):")
    print(f"   Min danger: {test_scores_arr.min():.3f}")
    print(f"   Max danger: {test_scores_arr.max():.3f}")
    print(f"   Mean danger: {test_scores_arr.mean():.3f}")
    print(f"   Median danger: {np.median(test_scores_arr):.3f}")
    
    from collections import Counter
    status_counts = Counter(test_statuses)
    print(f"\n📋 Status Distribution:")
    for status, count in status_counts.items():
        pct = count / len(test_statuses) * 100
        print(f"   {status}: {count}/50 ({pct:.1f}%)")
    
    # Quick histogram
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
    
    # Danger scores
    ax1.hist(test_scores_arr, bins=20, edgecolor='black', alpha=0.7)
    ax1.axvline(0.25, color='red', linestyle='--', linewidth=2, label='CRASH threshold')
    ax1.axvline(0.12, color='orange', linestyle='--', linewidth=2, label='CAUTION threshold')
    ax1.set_xlabel('Danger Score')
    ax1.set_ylabel('Frequency')
    ax1.set_title('Danger Score Distribution (50 frames)')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Status pie chart
    colors = {'SAFE': 'green', 'CAUTION': 'orange', 'CRASH RISK!': 'red'}
    status_colors = [colors.get(s, 'gray') for s in status_counts.keys()]
    ax2.pie(status_counts.values(), labels=status_counts.keys(), autopct='%1.1f%%',
            colors=status_colors, startangle=90)
    ax2.set_title('Safety Status Distribution')
    
    plt.tight_layout()
    plt.show()
    
    # Diagnosis
    print("\n" + "="*70)
    if test_scores_arr.mean() < 0.05:
        print("⚠️  WARNING: Scores still very low (mean < 0.05)")
        print("   → Try the NUCLEAR version with 5.0x multiplier")
        print("   → Or check if YOLO is detecting obstacles at all")
    elif test_scores_arr.mean() < 0.15:
        print("⚠️  Scores on the low side (mean < 0.15)")
        print("   → Consider increasing multiplier to 3.5x or 4.0x")
    elif test_scores_arr.mean() > 0.50:
        print("⚠️  Scores very high (mean > 0.50)")
        print("   → System might be too sensitive")
        print("   → Consider reducing multiplier to 2.0x")
    else:
        print("✅ Scores look reasonable (mean 0.15-0.50)")
        print("   → Should see mix of SAFE/CAUTION/CRASH RISK")
        print("   → Proceed with full video processing")
    print("="*70)
else:
    print("❌ Could not test - check video path")

print("\n✅ Cell complete! Proceed to Cell 12 to process full video.")

## **After Running This Cell:**


✅ AGGRESSIVE danger scoring loaded
   Key changes:
   - Vegetation weight: 1.2 (trees ARE obstacles)
   - Path blocking: 1.5x bonus
   - Multiple obstacles: up to 2.0x bonus
   - Overall multiplier: 2.5x
   - CRASH threshold: 0.25 (very sensitive)
   - CAUTION threshold: 0.12

📊 Testing on sample frames to verify sensitivity...

Sampling 50 frames...

📈 Quick Stats (50 frames):
   Min danger: 0.000
   Max danger: 1.000
   Mean danger: 0.461
   Median danger: 0.394

📋 Status Distribution:
   CRASH RISK!: 34/50 (68.0%)
   SAFE: 16/50 (32.0%)


<Figure size 1400x400 with 2 Axes>


✅ Scores look reasonable (mean 0.15-0.50)
   → Should see mix of SAFE/CAUTION/CRASH RISK
   → Proceed with full video processing

✅ Cell complete! Proceed to Cell 12 to process full video.


In [3]:
# ===== SAFETY ASSESSMENT FUNCTION (DEPTH-AWARE VERSION) =====
import numpy as np
import cv2
import matplotlib.pyplot as plt
from collections import Counter
from pathlib import Path
from tqdm.notebook import tqdm
def calculate_safety_from_detections(detections, frame_width, frame_height):
    """
    Depth-aware obstacle detection - solves the waterfall problem!
    
    Key improvements:
    - Uses object SIZE + POSITION as depth proxy
    - Large distant objects (waterfalls, mountains) = LOW danger
    - Small close objects (trees in path) = HIGH danger
    - Adaptive sensitivity based on scene depth
    
    Problem solved: Waterfall detected as "obstacle" but correctly
    identified as FAR AWAY, so danger score stays low.
    """
    
    if len(detections.boxes) == 0:
        return 0.0, "SAFE", []
    
    # ===== FLIGHT PATH DEFINITION =====
    path_x_min = frame_width * 0.35
    path_x_max = frame_width * 0.65
    path_y_min = frame_height * 0.50
    
    danger_score = 0.0
    dangerous_objects = []
    obstacles_in_path = 0
    depth_weights = []  # Track depth distribution
    
    for box in detections.boxes:
        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
        cls = int(box.cls[0].cpu().numpy())
        conf = float(box.conf[0].cpu().numpy())
        
        cx = (x1 + x2) / 2
        cy = (y1 + y2) / 2
        width = x2 - x1
        height = y2 - y1
        area = width * height
        size_ratio = area / (frame_width * frame_height)
        
        # ===== FILTER 1: Ignore very top of frame (distant background) =====
        if cy < path_y_min * 0.8:  # Top 40% of frame
            continue
        
        # ===== FILTER 2: Ignore tiny artifacts =====
        if size_ratio < 0.003:  # Less than 0.3% of frame
            continue
        
        # ===== DEPTH ESTIMATION (Heuristic) =====
        # Key insight: Large objects LOW in frame might be FAR AWAY
        # Small objects LOW in frame are CLOSE
        
        # Factor 1: Size-to-position ratio
        # Large object low = far (waterfall)
        # Small object low = close (tree trunk)
        y_position = cy / frame_height  # 0=top, 1=bottom
        
        # Depth heuristic (0 = far, 1 = near)
        if size_ratio > 0.19:  # Very large object
            # Likely distant scenery (waterfall, mountain, building)
            depth_weight = 0.2 * y_position  # Max 0.2 even at bottom
        elif size_ratio > 0.1:  # Large object
            # Could be close structure or far scenery
            depth_weight = 0.5 * y_position  # Moderate depth
        else:  # Normal/small object
            # Likely close obstacle (tree, person, vehicle)
            # Amplify by vertical position
            depth_weight = np.clip(size_ratio * 30 * y_position, 0.0, 1.0)
        
        depth_weights.append(depth_weight)
        
        # ===== SMART VEGETATION FILTERING =====
        if cls == 0:  # vegetation
            centrality = 1.0 - abs(cx - frame_width/2) / (frame_width/2)
            relative_height = (cy - path_y_min) / (frame_height - path_y_min)
            
            # Ignore background vegetation
            if size_ratio < 0.02 and centrality < 0.4 and relative_height < 0.3:
                continue
        
        # ===== PATH BLOCKING CHECK =====
        blocks_path = (x1 < path_x_max) and (x2 > path_x_min)
        if blocks_path:
            obstacles_in_path += 1
        
        # ===== DANGER FACTORS =====
        
        # Proximity (vertical position)
        proximity = np.clip((cy - path_y_min) / (frame_height - path_y_min), 0.2, 1.0)
        
        # Size factor
        size_factor = min(size_ratio * 15, 1.0)
        
        # Centrality (horizontal position)
        centrality = np.clip(1.0 - abs(cx - frame_width/2) / (frame_width/2), 0.2, 1.0)
        
        # Path blocking bonus
        path_bonus = 1.5 if blocks_path else 1.0
        
        # ===== CLASS WEIGHTS =====
        class_weight = {
            0: 1.2,   # vegetation (trees blocking path)
            1: 1.5,   # structure (walls, buildings)
            2: 0.5,   # fence (usually passable)
            3: 2.0,   # living (people, animals - MUST avoid)
            4: 1.5,   # vehicle (cars, bikes)
            5: 1.0,   # generic obstacle
        }.get(cls, 1.0)
        
        # ===== FINAL OBJECT DANGER (with depth weighting) =====
        base_danger = (
            proximity * 0.4 +
            size_factor * 0.4 +
            centrality * 0.2
        )
        
        # CRITICAL: Multiply by depth_weight
        # Far objects (depth_weight=0.2) contribute little
        # Near objects (depth_weight=0.8-1.0) contribute fully
        object_danger = (
            base_danger * class_weight * path_bonus * conf * depth_weight*1.1
        )
        
        danger_score += object_danger
        
        dangerous_objects.append({
            'bbox': (x1, y1, x2, y2),
            'class': cls,
            'danger': object_danger,
            'size': size_ratio,
            'depth_weight': depth_weight,
            'blocks_path': blocks_path,
            'proximity': proximity,
            'centrality': centrality
        })
    
    # ===== MULTI-OBSTACLE AMPLIFICATION =====
    if obstacles_in_path >= 5:
        danger_score *= 2.0
    elif obstacles_in_path >= 3:
        danger_score *= 1.5
    
    # ===== ADAPTIVE SENSITIVITY =====
    # If most objects are far away (low depth_weight), reduce overall sensitivity
    if depth_weights:
        mean_depth = np.mean(depth_weights)
        
        if mean_depth < 0.2:  # Mostly distant objects (scenery)
            sensitivity = 0.9  # Reduce sensitivity
        elif mean_depth > 0.6:  # Mostly close objects (real obstacles)
            sensitivity = 2.5  # Increase sensitivity
        else:  # Mixed scene
            sensitivity = 1.5
        
        danger_score *= sensitivity
    else:
        danger_score *= 2.5  # Default high sensitivity
    
    # Clamp to [0, 1]
    danger_score = np.clip(danger_score, 0.0, 1.0)
    
    # ===== STATUS DETERMINATION =====
    if danger_score > 0.30:
        status = "CRASH RISK!"
    elif danger_score > 0.15:
        status = "CAUTION"
    else:
        status = "SAFE"
    
    return danger_score, status, dangerous_objects


print("✅ Depth-aware safety assessment function defined")
print("\n📐 How it works:")
print("   - Estimates depth using SIZE + POSITION heuristic")
print("   - Large objects low in frame = FAR (waterfall, mountain)")
print("   - Small objects low in frame = NEAR (tree, person)")
print("   - Far objects get LOW depth weight (0.2)")
print("   - Near objects get HIGH depth weight (0.8-1.0)")
print("   - Adaptive sensitivity based on scene depth mix")
print("\n🎯 Result:")
print("   - Waterfall (large, low): depth_weight=0.2 → low danger")
print("   - Tree in path (small, low): depth_weight=0.9 → high danger")

# ===== LOAD YOLO MODEL =====

print("\n" + "="*70)
print("LOADING YOLO MODEL")
print("="*70)

from ultralytics import YOLO

yolo_weights = 'semantic_drone_training/yolov8n_semantic/weights/best.pt'

if Path(yolo_weights).exists():
    print(f"Loading trained YOLO from: {yolo_weights}")
    yolo = YOLO(yolo_weights)
    print("✅ Trained YOLO model loaded!")
else:
    print("⚠️  Trained weights not found, using pretrained COCO model")
    print(f"Expected path: {yolo_weights}")
    yolo = YOLO('yolov8n.pt')
    print("✅ Pretrained YOLO model loaded!")

# ===== QUICK TEST =====

print("\n" + "="*70)
print("📊 Testing depth-aware scoring...")
print("="*70)

test_cap = cv2.VideoCapture(video_path)
test_scores = []
test_statuses = []
test_depth_stats = []

print("\nSampling 50 frames...")
from tqdm.notebook import tqdm
for i in tqdm(range(50), desc="Testing"):
    ret, frame = test_cap.read()
    if not ret:
        break
    
    test_detections = yolo(frame, verbose=False)[0]
    danger, status, objs = calculate_safety_from_detections(
        test_detections, 
        int(test_cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
        int(test_cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    )
    test_scores.append(danger)
    test_statuses.append(status)
    
    if objs:
        mean_depth = np.mean([o['depth_weight'] for o in objs])
        test_depth_stats.append(mean_depth)

test_cap.release()

if test_scores:
    test_scores_arr = np.array(test_scores)
    
    print(f"\n📈 Quick Stats (50 frames):")
    print(f"   Min danger: {test_scores_arr.min():.3f}")
    print(f"   Max danger: {test_scores_arr.max():.3f}")
    print(f"   Mean danger: {test_scores_arr.mean():.3f}")
    print(f"   Median: {np.median(test_scores_arr):.3f}")
    
    if test_depth_stats:
        print(f"\n🔍 Depth Distribution:")
        print(f"   Mean depth weight: {np.mean(test_depth_stats):.3f}")
        print(f"   (0.0-0.3 = far scenery, 0.3-0.6 = mixed, 0.6-1.0 = close obstacles)")
    
    from collections import Counter
    status_counts = Counter(test_statuses)
    print(f"\n📋 Status Distribution:")
    for status, count in status_counts.items():
        pct = count / len(test_statuses) * 100
        print(f"   {status}: {count}/50 ({pct:.1f}%)")
    
    # Visualization
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    
    # Danger scores histogram
    axes[0].hist(test_scores_arr, bins=20, edgecolor='black', alpha=0.7, color='steelblue')
    axes[0].axvline(0.30, color='red', linestyle='--', linewidth=2, label='CRASH')
    axes[0].axvline(0.15, color='orange', linestyle='--', linewidth=2, label='CAUTION')
    axes[0].set_xlabel('Danger Score')
    axes[0].set_ylabel('Frequency')
    axes[0].set_title('Danger Score Distribution')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Status pie chart
    colors = {'SAFE': 'green', 'CAUTION': 'orange', 'CRASH RISK!': 'red'}
    status_colors = [colors.get(s, 'gray') for s in status_counts.keys()]
    axes[1].pie(status_counts.values(), labels=status_counts.keys(), 
                autopct='%1.1f%%', colors=status_colors, startangle=90)
    axes[1].set_title('Safety Status Distribution')
    
    # Depth weights histogram
    if test_depth_stats:
        axes[2].hist(test_depth_stats, bins=20, edgecolor='black', alpha=0.7, color='coral')
        axes[2].axvline(0.3, color='blue', linestyle='--', label='Far threshold')
        axes[2].axvline(0.6, color='green', linestyle='--', label='Near threshold')
        axes[2].set_xlabel('Mean Depth Weight')
        axes[2].set_ylabel('Frequency')
        axes[2].set_title('Depth Weight Distribution')
        axes[2].legend()
        axes[2].grid(True, alpha=0.3)
    else:
        axes[2].text(0.5, 0.5, 'No depth data', ha='center', va='center', fontsize=12)
        axes[2].set_title('Depth Weight Distribution')
    
    plt.tight_layout()
    plt.show()
    
    # Diagnosis
    print("\n" + "="*70)
    print("📊 DIAGNOSIS")
    print("="*70)
    
    if test_scores_arr.mean() < 0.05:
        print("⚠️  Scores very low (mean < 0.05)")
        print("   → Objects detected but danger too low")
        print("   → Try reducing depth_weight threshold or increasing sensitivity")
    elif test_scores_arr.mean() > 0.50:
        print("⚠️  Scores high (mean > 0.50)")
        print("   → System might be too sensitive")
        print("   → Try increasing depth_weight or reducing sensitivity")
    else:
        print("✅ Scores balanced (mean 0.05-0.50)")
        print("   → Depth-aware system working correctly!")
    
    if test_depth_stats:
        mean_d = np.mean(test_depth_stats)
        print(f"\n🔍 Scene Analysis:")
        if mean_d < 0.3:
            print(f"   → Mostly FAR objects (depth={mean_d:.2f})")
            print(f"   → System reducing sensitivity (0.8x)")
            print(f"   → Waterfalls/mountains correctly ignored")
        elif mean_d > 0.6:
            print(f"   → Mostly NEAR objects (depth={mean_d:.2f})")
            print(f"   → System increasing sensitivity (2.5x)")
            print(f"   → Close obstacles correctly flagged")
        else:
            print(f"   → MIXED scene (depth={mean_d:.2f})")
            print(f"   → System using moderate sensitivity (1.5x)")
    
    print("="*70)
else:
    print("❌ No test data collected - check video path")

print("\n✅ Cell complete! YOLO loaded and tested.")
print("📹 Proceed to Cell 12 to process full video.")

✅ Depth-aware safety assessment function defined

📐 How it works:
   - Estimates depth using SIZE + POSITION heuristic
   - Large objects low in frame = FAR (waterfall, mountain)
   - Small objects low in frame = NEAR (tree, person)
   - Far objects get LOW depth weight (0.2)
   - Near objects get HIGH depth weight (0.8-1.0)
   - Adaptive sensitivity based on scene depth mix

🎯 Result:
   - Waterfall (large, low): depth_weight=0.2 → low danger
   - Tree in path (small, low): depth_weight=0.9 → high danger

LOADING YOLO MODEL
Loading trained YOLO from: semantic_drone_training/yolov8n_semantic/weights/best.pt
✅ Trained YOLO model loaded!

📊 Testing depth-aware scoring...


NameError: name 'video_path' is not defined

In [5]:
# ===== SAFETY ASSESSMENT FUNCTION (DEPTH-AWARE VERSION) =====

def calculate_safety_from_detections(detections, frame_width, frame_height):
    """
    Depth-aware obstacle detection - solves the waterfall problem!
    
    Key improvements:
    - Uses object SIZE + POSITION as depth proxy
    - Large distant objects (waterfalls, mountains) = LOW danger
    - Small close objects (trees in path) = HIGH danger
    - Adaptive sensitivity based on scene depth
    
    Problem solved: Waterfall detected as "obstacle" but correctly
    identified as FAR AWAY, so danger score stays low.
    """
    
    if len(detections.boxes) == 0:
        return 0.0, "SAFE", []
    
    # ===== FLIGHT PATH DEFINITION =====
    path_x_min = frame_width * 0.35
    path_x_max = frame_width * 0.65
    path_y_min = frame_height * 0.50
    
    danger_score = 0.0
    dangerous_objects = []
    obstacles_in_path = 0
    depth_weights = []  # Track depth distribution
    
    for box in detections.boxes:
        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
        cls = int(box.cls[0].cpu().numpy())
        conf = float(box.conf[0].cpu().numpy())
        
        cx = (x1 + x2) / 2
        cy = (y1 + y2) / 2
        width = x2 - x1
        height = y2 - y1
        area = width * height
        size_ratio = area / (frame_width * frame_height)
        
        # ===== FILTER 1: Ignore very top of frame (distant background) =====
        if cy < path_y_min * 0.8:  # Top 40% of frame
            continue
        
        # ===== FILTER 2: Ignore tiny artifacts =====
        if size_ratio < 0.003:  # Less than 0.3% of frame
            continue
        
        # ===== DEPTH ESTIMATION (Heuristic) =====
        # Key insight: Large objects LOW in frame might be FAR AWAY
        # Small objects LOW in frame are CLOSE
        
        # Factor 1: Size-to-position ratio
        # Large object low = far (waterfall)
        # Small object low = close (tree trunk)
        y_position = cy / frame_height  # 0=top, 1=bottom
        
        # Depth heuristic (0 = far, 1 = near)
        if size_ratio > 0.15:  # Very large object
            # Likely distant scenery (waterfall, mountain, building)
            depth_weight = 0.2 * y_position  # Max 0.2 even at bottom
        elif size_ratio > 0.08:  # Large object
            # Could be close structure or far scenery
            depth_weight = 0.5 * y_position  # Moderate depth
        else:  # Normal/small object
            # Likely close obstacle (tree, person, vehicle)
            # Amplify by vertical position
            depth_weight = np.clip(size_ratio * 30 * y_position, 0.0, 1.0)
        
        depth_weights.append(depth_weight)
        
        # ===== SMART VEGETATION FILTERING =====
        if cls == 0:  # vegetation
            centrality = 1.0 - abs(cx - frame_width/2) / (frame_width/2)
            relative_height = (cy - path_y_min) / (frame_height - path_y_min)
            
            # Ignore background vegetation
            if size_ratio < 0.02 and centrality < 0.4 and relative_height < 0.3:
                continue
        
        # ===== PATH BLOCKING CHECK =====
        blocks_path = (x1 < path_x_max) and (x2 > path_x_min)
        if blocks_path:
            obstacles_in_path += 1
        
        # ===== DANGER FACTORS =====
        
        # Proximity (vertical position)
        proximity = np.clip((cy - path_y_min) / (frame_height - path_y_min), 0.2, 1.0)
        
        # Size factor
        size_factor = min(size_ratio * 15, 1.0)
        
        # Centrality (horizontal position)
        centrality = np.clip(1.0 - abs(cx - frame_width/2) / (frame_width/2), 0.2, 1.0)
        
        # Path blocking bonus
        path_bonus = 1.5 if blocks_path else 1.0
        
        # ===== CLASS WEIGHTS =====
        class_weight = {
            0: 1.2,   # vegetation (trees blocking path)
            1: 1.5,   # structure (walls, buildings)
            2: 0.5,   # fence (usually passable)
            3: 2.0,   # living (people, animals - MUST avoid)
            4: 1.5,   # vehicle (cars, bikes)
            5: 1.0,   # generic obstacle
        }.get(cls, 1.0)
        
        # ===== FINAL OBJECT DANGER (with depth weighting) =====
        base_danger = (
            proximity * 0.4 +
            size_factor * 0.4 +
            centrality * 0.2
        )
        
        # CRITICAL: Multiply by depth_weight
        # Far objects (depth_weight=0.2) contribute little
        # Near objects (depth_weight=0.8-1.0) contribute fully
        object_danger = (
            base_danger * class_weight * path_bonus * conf * depth_weight
        )
        
        danger_score += object_danger
        
        dangerous_objects.append({
            'bbox': (x1, y1, x2, y2),
            'class': cls,
            'danger': object_danger,
            'size': size_ratio,
            'depth_weight': depth_weight,
            'blocks_path': blocks_path,
            'proximity': proximity,
            'centrality': centrality
        })
    
    # ===== MULTI-OBSTACLE AMPLIFICATION =====
    if obstacles_in_path >= 5:
        danger_score *= 2.0
    elif obstacles_in_path >= 3:
        danger_score *= 1.5
    
    # ===== ADAPTIVE SENSITIVITY =====
    # If most objects are far away (low depth_weight), reduce overall sensitivity
    if depth_weights:
        mean_depth = np.mean(depth_weights)
        
        if mean_depth < 0.3:  # Mostly distant objects (scenery)
            sensitivity = 0.8  # Reduce sensitivity
        elif mean_depth > 0.6:  # Mostly close objects (real obstacles)
            sensitivity = 2.5  # Increase sensitivity
        else:  # Mixed scene
            sensitivity = 1.5
        
        danger_score *= sensitivity
    else:
        danger_score *= 2.5  # Default high sensitivity
    
    # Clamp to [0, 1]
    danger_score = np.clip(danger_score, 0.0, 1.0)
    
    # ===== STATUS DETERMINATION =====
    if danger_score > 0.30:
        status = "CRASH RISK!"
    elif danger_score > 0.15:
        status = "CAUTION"
    else:
        status = "SAFE"
    
    return danger_score, status, dangerous_objects


print("✅ Depth-aware safety assessment loaded")
print("\n📐 How it works:")
print("   - Estimates depth using SIZE + POSITION heuristic")
print("   - Large objects low in frame = FAR (waterfall, mountain)")
print("   - Small objects low in frame = NEAR (tree, person)")
print("   - Far objects get LOW depth weight (0.2)")
print("   - Near objects get HIGH depth weight (0.8-1.0)")
print("   - Adaptive sensitivity based on scene depth mix")
print("\n🎯 Result:")
print("   - Waterfall (large, low): depth_weight=0.2 → low danger")
print("   - Tree in path (small, low): depth_weight=0.9 → high danger")

# ===== QUICK TEST =====

print("\n" + "="*70)
print("📊 Testing depth-aware scoring...")
print("="*70)

test_cap = cv2.VideoCapture(video_path)
test_scores = []
test_statuses = []
test_depth_stats = []

print("\nSampling 50 frames...")
for i in range(50):
    ret, frame = test_cap.read()
    if not ret:
        break
    
    test_detections = yolo(frame, verbose=False)[0]
    danger, status, objs = calculate_safety_from_detections(
        test_detections, 
        int(test_cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
        int(test_cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    )
    test_scores.append(danger)
    test_statuses.append(status)
    
    if objs:
        mean_depth = np.mean([o['depth_weight'] for o in objs])
        test_depth_stats.append(mean_depth)

test_cap.release()

if test_scores:
    test_scores_arr = np.array(test_scores)
    
    print(f"\n📈 Quick Stats (50 frames):")
    print(f"   Min danger: {test_scores_arr.min():.3f}")
    print(f"   Max danger: {test_scores_arr.max():.3f}")
    print(f"   Mean danger: {test_scores_arr.mean():.3f}")
    print(f"   Median: {np.median(test_scores_arr):.3f}")
    
    if test_depth_stats:
        print(f"\n🔍 Depth Distribution:")
        print(f"   Mean depth weight: {np.mean(test_depth_stats):.3f}")
        print(f"   (0.0-0.3 = far scenery, 0.3-0.6 = mixed, 0.6-1.0 = close obstacles)")
    
    status_counts = Counter(test_statuses)
    print(f"\n📋 Status Distribution:")
    for status, count in status_counts.items():
        pct = count / len(test_statuses) * 100
        print(f"   {status}: {count}/50 ({pct:.1f}%)")
    
    # Visualization
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    
    # Danger scores histogram
    axes[0].hist(test_scores_arr, bins=20, edgecolor='black', alpha=0.7, color='steelblue')
    axes[0].axvline(0.30, color='red', linestyle='--', linewidth=2, label='CRASH')
    axes[0].axvline(0.15, color='orange', linestyle='--', linewidth=2, label='CAUTION')
    axes[0].set_xlabel('Danger Score')
    axes[0].set_ylabel('Frequency')
    axes[0].set_title('Danger Score Distribution')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Status pie chart
    colors = {'SAFE': 'green', 'CAUTION': 'orange', 'CRASH RISK!': 'red'}
    status_colors = [colors.get(s, 'gray') for s in status_counts.keys()]
    axes[1].pie(status_counts.values(), labels=status_counts.keys(), 
                autopct='%1.1f%%', colors=status_colors, startangle=90)
    axes[1].set_title('Safety Status Distribution')
    
    # Depth weights histogram
    if test_depth_stats:
        axes[2].hist(test_depth_stats, bins=20, edgecolor='black', alpha=0.7, color='coral')
        axes[2].axvline(0.3, color='blue', linestyle='--', label='Far threshold')
        axes[2].axvline(0.6, color='green', linestyle='--', label='Near threshold')
        axes[2].set_xlabel('Mean Depth Weight')
        axes[2].set_ylabel('Frequency')
        axes[2].set_title('Depth Weight Distribution')
        axes[2].legend()
        axes[2].grid(True, alpha=0.3)
    else:
        axes[2].text(0.5, 0.5, 'No depth data', ha='center', va='center')
        axes[2].set_title('Depth Weight Distribution')
    
    plt.tight_layout()
    plt.show()
    
    # Diagnosis
    print("\n" + "="*70)
    if test_scores_arr.mean() < 0.05:
        print("⚠️  Scores very low - might need further tuning")
    elif test_scores_arr.mean() > 0.50:
        print("⚠️  Scores high - system might be too sensitive")
    else:
        print("✅ Scores balanced - depth-aware system working!")
        print("   Waterfall/scenery should show LOW danger")
        print("   Close obstacles should show HIGH danger")
    print("="*70)

print("\n✅ Cell complete! Proceed to Cell 12.")

✅ Depth-aware safety assessment loaded

📐 How it works:
   - Estimates depth using SIZE + POSITION heuristic
   - Large objects low in frame = FAR (waterfall, mountain)
   - Small objects low in frame = NEAR (tree, person)
   - Far objects get LOW depth weight (0.2)
   - Near objects get HIGH depth weight (0.8-1.0)
   - Adaptive sensitivity based on scene depth mix

🎯 Result:
   - Waterfall (large, low): depth_weight=0.2 → low danger
   - Tree in path (small, low): depth_weight=0.9 → high danger

📊 Testing depth-aware scoring...

Sampling 50 frames...


NameError: name 'yolo' is not defined

In [9]:
# ===== LOAD DRONET (if not already loaded) =====

try:
    # Check if dronet is already loaded
    dronet.summary()
    print("✅ DroNet already loaded")
except:
    print("Loading DroNet...")
    from tensorflow.keras.models import model_from_json
    
    model_json_path = r"of-obstacledetection\DroNeTello\models\DroNet\model_struct.json"
    model_weights_path = r"of-obstacledetection\DroNeTello\models\DroNet\model_weights_new_best.h5"
    
    with open(model_json_path, "r") as f:
        dronet = model_from_json(f.read())
    dronet.load_weights(model_weights_path)
    print("✅ DroNet loaded!")

# Define preprocessing if not defined
def preprocess_dronet(frame):
    """Preprocess frame for DroNet"""
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    resized = cv2.resize(gray, (200, 200))
    normalized = resized / 255.0
    return np.expand_dims(normalized, axis=(0, -1))

print("✅ DroNet and preprocessing ready")

# === YOUR EXISTING VIDEO PROCESSING CODE STARTS HERE ===




# ===== HYBRID OBSTACLE AVOIDANCE SYSTEM =====
import cv2
import numpy as np
from tqdm import tqdm
from pathlib import Path

print("="*70)
print("STEP 4: PROCESSING VIDEO WITH HYBRID SYSTEM (STABLE VERSION)")
print("="*70)

# Paths
video_path = r"C:\Users\vedhr\CODES\Obstacle avoidance\Fpv44.mp4"
output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)
output_path = output_dir / "hybrid_outputnew4.mp4"

cap = cv2.VideoCapture(video_path)
if not cap.isOpened():
    raise FileNotFoundError(f"❌ Unable to open video: {video_path}")

fps = int(cap.get(cv2.CAP_PROP_FPS)) or 30
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

# Safer codec
fourcc = cv2.VideoWriter_fourcc(*'XVID')
out = cv2.VideoWriter(str(output_path), fourcc, fps, (width, height))

print(f"📹 Processing: {video_path}")
print(f"📊 {total_frames} frames at {fps} FPS")
print(f"📐 Resolution: {width}x{height}")
print(f"💾 Output: {output_path}\n")

frame_count = 0
success_count = 0

for _ in tqdm(range(total_frames), desc="Processing"):
    ret, frame = cap.read()
    if not ret:
        print(f"⚠️ Frame read failed at {frame_count}, stopping early.")
        break
    frame_count += 1

    # === YOLO + DRONET Processing ===
    dronet_input = preprocess_dronet(frame)
    pred = dronet.predict(dronet_input, verbose=0)[0]
    steering = float(pred[0])

    detections = yolo(frame, verbose=False)[0]
    crash_prob, status, dangerous_objs = calculate_safety_from_detections(detections, width, height)

    annotated = detections.plot()

    # Ensure frame matches writer resolution
    if annotated.shape[1] != width or annotated.shape[0] != height:
        annotated = cv2.resize(annotated, (width, height))

    # Overlays
    cv2.rectangle(annotated, (5, 5), (400, 130), (0, 0, 0), -1)
    cv2.rectangle(annotated, (5, 5), (400, 130), (255, 255, 255), 2)

    cv2.putText(annotated, f"Steering: {steering:.3f}", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

    crash_color = (0, 255, 0) if crash_prob < 0.25 else (0, 165, 255) if crash_prob < 0.5 else (0, 0, 255)
    cv2.putText(annotated, f"Crash Prob: {crash_prob:.3f}", (10, 60),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, crash_color, 2)

    status_color = (0, 255, 0) if status == "SAFE" else (0, 165, 255) if status == "CAUTION" else (0, 0, 255)
    cv2.putText(annotated, f"Status: {status}", (10, 90),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, status_color, 2)

    cv2.putText(annotated, "DroNet + YOLO", (10, 120),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)

    # Steering arrow
    arrow_y = height - 60
    cx = width // 2
    end_x = int(cx + steering * 250)
    cv2.line(annotated, (50, arrow_y), (width - 50, arrow_y), (100, 100, 100), 2)
    cv2.arrowedLine(annotated, (cx, arrow_y), (end_x, arrow_y), (0, 255, 255), 5, tipLength=0.3)

    out.write(annotated)
    success_count += 1

cap.release()
out.release()
cv2.destroyAllWindows()

print("\n✅ Processing complete!")
print(f"📁 Output saved to: {output_path}")
print(f"📊 Frames processed: {success_count}/{frame_count}")


Model: "model_1"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_1 (InputLayer)        [(None, 200, 200, 1)]        0         []                            
                                                                                                  
 conv2d_1 (Conv2D)           (None, 100, 100, 32)         832       ['input_1[0][0]']             
                                                                                                  
 max_pooling2d_1 (MaxPoolin  (None, 49, 49, 32)           0         ['conv2d_1[0][0]']            
 g2D)                                                                                             
                                                                                                  
 batch_normalization_1 (Bat  (None, 49, 49, 32)           128       ['max_pooling2d_1[0][0]'

Processing: 100%|████████████████████████████████████████████████████████████████████| 467/467 [01:33<00:00,  4.97it/s]


✅ Processing complete!
📁 Output saved to: outputs\hybrid_outputnew4.mp4
📊 Frames processed: 467/467


In [15]:
# ===== LOAD DRONET (if not already loaded) =====

try:
    # Check if dronet is already loaded
    dronet.summary()
    print("✅ DroNet already loaded")
except:
    print("Loading DroNet...")
    from tensorflow.keras.models import model_from_json
    
    model_json_path = r"of-obstacledetection\DroNeTello\models\DroNet\model_struct.json"
    model_weights_path = r"of-obstacledetection\DroNeTello\models\DroNet\model_weights_new_best.h5"
    
    with open(model_json_path, "r") as f:
        dronet = model_from_json(f.read())
    dronet.load_weights(model_weights_path)
    print("✅ DroNet loaded!")

# Define preprocessing if not defined
def preprocess_dronet(frame):
    """Preprocess frame for DroNet"""
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    resized = cv2.resize(gray, (200, 200))
    normalized = resized / 255.0
    return np.expand_dims(normalized, axis=(0, -1))

print("✅ DroNet and preprocessing ready")

# === YOUR EXISTING VIDEO PROCESSING CODE STARTS HERE ===

# ===== HYBRID OBSTACLE AVOIDANCE SYSTEM =====
import cv2
import numpy as np
from tqdm import tqdm
from pathlib import Path

print("="*70)
print("STEP 4: PROCESSING VIDEO WITH HYBRID SYSTEM (STABLE VERSION)")
print("="*70)

# Paths
video_path = r"C:\Users\vedhr\CODES\Obstacle avoidance\istockphoto-1397265636-640_adpp_is.mp4"
output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)
output_path = output_dir / "hybrid_outputnew55.mp4"

cap = cv2.VideoCapture(video_path)
if not cap.isOpened():
    raise FileNotFoundError(f"❌ Unable to open video: {video_path}")

fps = int(cap.get(cv2.CAP_PROP_FPS)) or 30
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

# Safer codec
fourcc = cv2.VideoWriter_fourcc(*'XVID')
out = cv2.VideoWriter(str(output_path), fourcc, fps, (width, height))

print(f"📹 Processing: {video_path}")
print(f"📊 {total_frames} frames at {fps} FPS")
print(f"📐 Resolution: {width}x{height}")
print(f"💾 Output: {output_path}\n")

frame_count = 0
success_count = 0

for _ in tqdm(range(total_frames), desc="Processing"):
    ret, frame = cap.read()
    if not ret:
        print(f"⚠️ Frame read failed at {frame_count}, stopping early.")
        break
    frame_count += 1

    # === YOLO + DRONET Processing ===
    dronet_input = preprocess_dronet(frame)
    pred = dronet.predict(dronet_input, verbose=0)[0]
    steering = float(pred[0])

    detections = yolo(frame, verbose=False)[0]
    crash_prob, status, dangerous_objs = calculate_safety_from_detections(detections, width, height)

    annotated = detections.plot()

    # Ensure frame matches writer resolution
    if annotated.shape[1] != width or annotated.shape[0] != height:
        annotated = cv2.resize(annotated, (width, height))

    # Overlays
    cv2.rectangle(annotated, (5, 5), (400, 130), (0, 0, 0), -1)
    cv2.rectangle(annotated, (5, 5), (400, 130), (255, 255, 255), 2)

    cv2.putText(annotated, f"Steering: {steering:.3f}", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

    crash_color = (0, 255, 0) if crash_prob < 0.25 else (0, 165, 255) if crash_prob < 0.5 else (0, 0, 255)
    cv2.putText(annotated, f"Crash Prob: {crash_prob:.3f}", (10, 60),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, crash_color, 2)

    status_color = (0, 255, 0) if status == "SAFE" else (0, 165, 255) if status == "CAUTION" else (0, 0, 255)
    cv2.putText(annotated, f"Status: {status}", (10, 90),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, status_color, 2)

    cv2.putText(annotated, "DroNet + YOLO", (10, 120),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)

    # ===== Enhanced 2D Steering Visualization (New) =====
    steering_x = steering  # horizontal from DroNet
    steering_y = (0.5 - crash_prob) * 2  # pseudo vertical control (up if high risk)

    arrow_center = (width // 2, height - 100)
    arrow_length = 150

    end_x = int(arrow_center[0] + steering_x * arrow_length)
    end_y = int(arrow_center[1] - steering_y * arrow_length)

    # Draw cross axes
    cv2.line(annotated, (arrow_center[0] - arrow_length, arrow_center[1]),
             (arrow_center[0] + arrow_length, arrow_center[1]), (100, 100, 100), 2)
    cv2.line(annotated, (arrow_center[0], arrow_center[1] + arrow_length),
             (arrow_center[0], arrow_center[1] - arrow_length), (100, 100, 100), 2)

    # Draw steering arrow
    cv2.arrowedLine(annotated, arrow_center, (end_x, end_y),
                    (0, 255, 255), 5, tipLength=0.3)

    # Show numeric steering values
    cv2.putText(annotated, f"Steer X: {steering_x:+.2f}", (10, height - 40),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
    cv2.putText(annotated, f"Steer Y: {steering_y:+.2f}", (10, height - 10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
    # ==============================================

    out.write(annotated)
    success_count += 1

cap.release()
out.release()
cv2.destroyAllWindows()

print("\n✅ Processing complete!")
print(f"📁 Output saved to: {output_path}")
print(f"📊 Frames processed: {success_count}/{frame_count}")


Model: "model_1"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_1 (InputLayer)        [(None, 200, 200, 1)]        0         []                            
                                                                                                  
 conv2d_1 (Conv2D)           (None, 100, 100, 32)         832       ['input_1[0][0]']             
                                                                                                  
 max_pooling2d_1 (MaxPoolin  (None, 49, 49, 32)           0         ['conv2d_1[0][0]']            
 g2D)                                                                                             
                                                                                                  
 batch_normalization_1 (Bat  (None, 49, 49, 32)           128       ['max_pooling2d_1[0][0]'

Processing: 100%|████████████████████████████████████████████████████████████████████| 300/300 [01:38<00:00,  3.06it/s]


✅ Processing complete!
📁 Output saved to: outputs\hybrid_outputnew55.mp4
📊 Frames processed: 300/300
